# GameScope: Global Video Game Sales Intelligence

This notebook presents the university EDA using the cleaned **`vgsales.csv`** dataset as the primary source. The 2024 vgchartz dataset has different coverage and schema; it is not merged into this analysis.

For each section, the markdown describes what is analyzed and why. The code computes and displays the findings from the processed source, and plots explain the pattern being examined. No result is typed into the notebook as a fixed value.

**Reproduce:** run the cleaning pipeline, then `python -m python.analytics_pipeline`, then execute all notebook cells from the project root. Install the packages listed in `python/requirements.txt` first.


## 1. Load cleaned data and review data quality

**What and why:** Check the cleaned table's shape, fields, missingness and duplicates before interpreting summaries. This makes the analytical base visible and keeps cleaning choices auditable.

**Finding:** The cell reports dataset counts and missing values directly from the cleaned CSV. Cleaning details and transformations are documented in `reports/data-cleaning.md`.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from python.load_data import load_primary_cleaned
from python.analytics_pipeline import build_analytics

rows = load_primary_cleaned()
df = pd.DataFrame(rows)
for field in ("Rank", "Year", "NA_Sales", "EU_Sales", "JP_Sales", "Other_Sales", "Global_Sales"):
    if field in df.columns:
        df[field] = pd.to_numeric(df[field], errors="coerce")
results = build_analytics(rows)
print(f"Cleaned primary data: {len(df):,} rows ? {len(df.columns)} columns")
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True),
})
display(quality)
print(f"Exact duplicate rows: {int(df.duplicated().sum()):,}")


## 2. Descriptive statistics and top 10 games

**What and why:** Describe the numeric sales fields and identify the ten records with the highest reported global sales. This establishes scale and highlights the most commercially successful game records.

**Visualization:** A horizontal bar chart compares the top records. Long-tail sales and ties are easier to inspect in a ranked table than from a single average.


In [ ]:
display(pd.DataFrame(results["descriptive_statistics"]).T)
top10 = pd.DataFrame(results["top_games"])
display(top10)
if not top10.empty:
    labels = top10["name"].astype(str) + " (" + top10["platform"].astype(str) + ")"
    ax = top10.assign(label=labels).sort_values("global_sales").plot.barh(x="label", y="global_sales", legend=False, figsize=(10, 6), color="#4976a8")
    ax.set(title="Top 10 game records by global sales", xlabel="Global sales (source units)", ylabel="Game and platform")
    plt.tight_layout()
    plt.show()


## 3. Releases and global sales over time

**What and why:** Count records by release year and aggregate global sales by year. Release counts describe the dataset's observed production history; sales totals show the scale recorded for those releases.

**Visualization:** Two aligned time series keep volume of releases distinct from sales totals. Invalid or missing years are excluded from yearly grouping and are reported in data-quality outputs.


In [ ]:
annual = pd.DataFrame(results["annual_trends"])
display(annual)
if not annual.empty:
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    axes[0].plot(annual["year"], annual["release_count"], color="#4976a8")
    axes[0].set_ylabel("Game records")
    axes[0].set_title("Game releases and global sales by year")
    axes[1].plot(annual["year"], annual["global_sales_total"], color="#d27c35")
    axes[1].set_ylabel("Global sales (source units)")
    axes[1].set_xlabel("Release year")
    plt.tight_layout()
    plt.show()


## 4. Genre frequency and global sales

**What and why:** Compare record counts with aggregate global sales by genre. These answer different questions: how often a genre appears and how much reported sales its records accumulate.

**Visualization:** Paired horizontal bars make differences in genre representation and total sales visible without treating frequency as a proxy for commercial performance.


In [ ]:
genre_counts = pd.DataFrame(results["genre_frequency"])
genre_sales = pd.DataFrame(results["genre_global_sales"])
display(genre_counts)
display(genre_sales)
if not genre_counts.empty and not genre_sales.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    genre_counts.sort_values("game_count").plot.barh(x="genre", y="game_count", legend=False, ax=axes[0], color="#4976a8")
    axes[0].set(title="Records by genre", xlabel="Game records", ylabel="Genre")
    genre_sales.sort_values("global_sales_total").plot.barh(x="genre", y="global_sales_total", legend=False, ax=axes[1], color="#d27c35")
    axes[1].set(title="Global sales by genre", xlabel="Global sales (source units)", ylabel="Genre")
    plt.tight_layout()
    plt.show()


## 5. Platform counts, sales, and evolution

**What and why:** Compare platform catalog size and total global sales, then examine how releases and sales vary by platform-year. Platforms have different life spans, so totals alone may hide their timing.

**Visualization:** The top platforms are shown for readable comparison; a platform-year heatmap reveals the timing and concentration of observed sales. The full tables remain available for exploration.


In [ ]:
platform_counts = pd.DataFrame(results["platform_game_counts"])
platform_sales = pd.DataFrame(results["platform_global_sales"])
evolution = pd.DataFrame(results["platform_evolution"])
display(platform_counts.head(15))
display(platform_sales.head(15))
if not platform_sales.empty:
    platform_sales.head(15).sort_values("global_sales_total").plot.barh(x="platform", y="global_sales_total", legend=False, figsize=(9, 6), color="#4976a8")
    plt.title("Top platforms by global sales")
    plt.xlabel("Global sales (source units)")
    plt.tight_layout()
    plt.show()
if not evolution.empty:
    pivot = evolution.pivot(index="platform", columns="year", values="global_sales_total").fillna(0)
    top_platforms = platform_sales.head(20)["platform"]
    pivot = pivot.reindex(top_platforms).dropna(how="all")
    if not pivot.empty:
        fig, ax = plt.subplots(figsize=(14, max(5, len(pivot) * .3)))
        image = ax.imshow(pivot, aspect="auto", interpolation="nearest", cmap="YlOrRd")
        ax.set_yticks(range(len(pivot)), labels=pivot.index)
        ax.set_xticks(range(0, len(pivot.columns), max(1, len(pivot.columns)//12)), labels=pivot.columns[::max(1, len(pivot.columns)//12)], rotation=45)
        ax.set(title="Platform global sales by release year (top platforms)", xlabel="Year", ylabel="Platform")
        fig.colorbar(image, ax=ax, label="Global sales")
        plt.tight_layout()
        plt.show()


## 6. Regional sales and genre preferences

**What and why:** Compare North America, Europe, Japan and Other sales, then examine genre preference within each region. Preference affinity is a ratio of a genre's share within the region to that genre's overall share across regional sales; values above one indicate relative over-indexing.

**Visualization:** The regional totals show market scale; a genre-by-region heatmap of affinity highlights relative differences while the accompanying table reports underlying sales and shares.


In [ ]:
regional = pd.DataFrame(results["regional_sales_comparison"])
preferences = pd.DataFrame(results["regional_genre_preferences"])
display(regional)
display(preferences)
if not regional.empty:
    regional.plot.bar(x="region", y="sales_total", legend=False, figsize=(8, 4), color="#4976a8")
    plt.title("Reported sales by region")
    plt.ylabel("Sales (source units)")
    plt.tight_layout()
    plt.show()
if not preferences.empty:
    heat = preferences.pivot(index="genre", columns="region", values="affinity_index")
    fig, ax = plt.subplots(figsize=(9, max(5, len(heat) * .35)))
    image = ax.imshow(heat.fillna(0), aspect="auto", cmap="RdBu_r", vmin=0, vmax=max(2, heat.max().max()))
    ax.set_xticks(range(len(heat.columns)), labels=heat.columns)
    ax.set_yticks(range(len(heat.index)), labels=heat.index)
    ax.set_title("Regional genre affinity (1 = overall baseline)")
    fig.colorbar(image, ax=ax, label="Affinity index")
    plt.tight_layout()
    plt.show()


## 7. Publisher intelligence

**What and why:** Rank publishers by total global sales, summarize catalogue breadth and identify publishers with a large share of sales in a given genre. A genre share above the configurable threshold indicates a strong contribution to that genre's observed market sales, not exclusivity.

**Visualization:** The chart compares top publisher sales; the table exposes genre dominance and the threshold used.


In [ ]:
publishers = pd.DataFrame(results["top_publishers"])
dominance = pd.DataFrame(results["publisher_genre_dominance"])
display(publishers)
display(dominance.sort_values("genre_market_share", ascending=False).head(25))
if not publishers.empty:
    publishers.sort_values("global_sales_total").plot.barh(x="publisher", y="global_sales_total", legend=False, figsize=(10, 6), color="#4976a8")
    plt.title("Top publishers by global sales")
    plt.xlabel("Global sales (source units)")
    plt.tight_layout()
    plt.show()


## 8. Market Power Index

**What and why:** Explore a project-defined publisher composite combining sales share, genre leadership coverage, genre diversity, regional consistency and hit rate. It is a descriptive index designed for this project, not a standard industry measure.

**Visualization:** The ranked bar chart shows the composite alongside a component table. Interpret publisher comparisons with the component definitions and weights in `data/processed/analytics/README.md`; coverage and catalogue size affect the score.


In [ ]:
mpi = pd.DataFrame(results["market_power_index"])
display(mpi.head(20))
if not mpi.empty:
    mpi.head(15).sort_values("market_power_index").plot.barh(x="publisher", y="market_power_index", legend=False, figsize=(10, 6), color="#8b6cab")
    plt.title("Market Power Index: leading publishers")
    plt.xlabel("Composite score (0?1)")
    plt.tight_layout()
    plt.show()


## 9. Regional Preference Divergence

**What and why:** Compare genre mixes across regions using total variation distance, track each region's top genre by year, and identify games with unusually concentrated regional sales. Divergence is descriptive and reflects the source's reported regional totals.

**Visualization:** A yearly divergence line shows how far regional genre mixes differ on average. The regional favorites table and outlier table provide context; missing or zero regional totals are excluded only where the metric cannot be computed.


In [ ]:
divergence = pd.DataFrame(results["regional_preference_divergence_by_year"])
favorites = pd.DataFrame(results["regional_favorites_by_year"])
stability = pd.DataFrame(results["regional_preference_stability"])
outliers = pd.DataFrame(results["cultural_outliers"])
display(favorites)
display(stability)
display(outliers.head(25))
if not divergence.empty:
    divergence.plot(x="year", y="mean_pairwise_divergence", legend=False, figsize=(11, 4), color="#d27c35")
    plt.title("Average pairwise regional genre divergence by year")
    plt.ylabel("Total variation distance (0?1)")
    plt.xlabel("Year")
    plt.tight_layout()
    plt.show()


## 10. Evidence-based insights and conclusions

**What and why:** The insight cards summarize computed metrics with the source and coverage used, providing a concise bridge from analysis to storytelling. The final notes state methodological limits alongside observed findings.

**Conclusion guidance:** Describe patterns visible in the displayed tables and charts, distinguish counts from sales, note missing/invalid years and regional coverage, and avoid causal claims. Duplicate rows and historical sales definitions constrain what these data can establish.


In [ ]:
for card in results["insights"]:
    print(f"{card['title']}: {card['text']}")
    print(f"  Metric: {card['metric']} | Value: {card['value']} | Coverage: {card.get('coverage')} | Source: {card.get('source')}")


## Data-wrangling challenges and limitations

- The primary source has missing metadata and year values; group-based analyses use only records eligible for the relevant field, without inventing values.
- Sales are source-reported estimates. This notebook does not infer inflation adjustment, platform equivalence, or current market share.
- Publisher/game naming and repeated titles can complicate identity matching. The core assignment analyses retain source records and do not deduplicate game identities automatically.
- Regional sales fields do not fully describe every market. The `Other` field is an aggregate, and regional percentages use reported values only.
- The vgchartz 2024 source is intentionally kept outside these assignment summaries because its coverage and schema differ. Any advanced comparison should be defined explicitly before matching records.

The concrete findings are produced by running the cells above against the current cleaned dataset; no fixed numerical findings have been inserted into this notebook.
